# Demo 09e — Separating a mixture, three ways

**Class 09 · E1** (slides: `slides/09_blind_signal_separation.md`)

The running example of the class: a 3 Hz sine and a 7 Hz chirp mixed by a 2 x 2 matrix.
Three decompositions, three views of the same cloud: PCA (variance, uncorrelated),
ICA (non-Gaussianity, independent), NMF (positivity, additive). One table, one conclusion.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from sklearn.decomposition import PCA, FastICA

from faa import bss, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")

## Slide: The mixture, the three methods, one table

PCA and ICA recover the sources by unmixing the rows; NMF needs non-negative data, so
the mixtures are shifted back to positive and the recovered components are matched.
`bss.match` pairs each estimate with its true source and fixes the sign.

In [ ]:
X, S, A = bss.mixtures(n=300, seed=SEED)
pca = PCA(2).fit(X.T)
S_pca = bss.match(S, pca.transform(X.T).T)[0]  # scores: the projected sensor data, one row per component
ica = FastICA(2, random_state=SEED).fit(X.T)
S_ica = bss.match(S, ica.transform(X.T).T)[0]
rec, W, H, _ = pynnmf.nmf_mu(X - X.min() + 0.05, 2, n=500, seed=SEED)  # shift: NMF needs X >= 0
S_nmf = bss.match(S, H)[0]
slide("Three separations of one 2-sensor mixture")
rows = []
for name, est in [("PCA", S_pca), ("ICA", S_ica), ("NMF (shifted)", S_nmf)]:
    for i in range(2):
        c = max(abs(np.corrcoef(est[j], S[i])[0, 1]) for j in range(2))  # match blind: best of the two estimates
        rows.append(
            {
                "method": name,
                "source": f"s{i + 1}",
                "corr": c,
                "SNR (dB)": 10 * np.log10(max(c**2, 1e-12) / (1 - min(c**2, 1 - 1e-12))),
            }
        )
print(
    pl.DataFrame(rows)
    .with_columns(pl.col("corr").round(3), pl.col("SNR (dB)").round(1))
    .pivot(values=["corr", "SNR (dB)"], on="source", separator=" ", index="method")
)

## Slide: What each method assumes — the constraint map

The same scatter plot, three views: PCA finds the axes of variance, ICA the axes of
independence (kurtosis), NMF the positive cone. The figure is the argument.

In [ ]:
f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 2.4))
ax[0].scatter(X[0], X[1], s=3, color=GREY)
for v in pca.components_:
    ax[0].axline((0, 0), slope=v[1] / v[0], color=BLUE, lw=1)
ax[0].set_title("PCA axes", fontsize=8)
ax[1].scatter(X[0], X[1], s=3, color=GREY)
for v in ica.components_:
    ax[1].axline((0, 0), slope=v[1] / v[0], color=ORANGE, lw=1)
ax[1].set_title("ICA axes", fontsize=8)
Xs = X - X.min() + 0.05
ax[2].scatter(Xs[0], Xs[1], s=3, color=GREY)
for j in range(2):  # the NMF axes: the columns of W are the edges of the cone that holds the data
    wj = W[:, j] / np.linalg.norm(W[:, j])
    w = wj * min(Xs[0].max() / wj[0], Xs[1].max() / wj[1]) * 0.97  # as long as the cone edge inside the plot
    ax[2].annotate("", xy=w, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=GREEN, lw=1.4))
    ax[2].text(w[0], w[1], f"$w_{j + 1}$", color=GREEN, fontsize=8)
ax[2].set_xlim(0, Xs[0].max() * 1.05)
ax[2].set_ylim(0, Xs[1].max() * 1.05)
ax[2].set_title("NMF axes ($W$)", fontsize=8)
for a in ax:
    a.set_xlabel("sensor 1", fontsize=8), a.set_ylabel("sensor 2", fontsize=8)
f.subplots_adjust(wspace=0.45)
plotlib.save(f, "p5_three_views")
print("PCA directions are orthogonal; ICA directions are not; NMF axes are the cone edges (columns of W, all >= 0).")
print(
    f"angle between the NMF axes: {np.degrees(np.arccos(np.clip((W[:, 0] / np.linalg.norm(W[:, 0])) @ (W[:, 1] / np.linalg.norm(W[:, 1])), -1, 1))):.1f} deg (PCA axes: 90.0 deg)"
)

## Slide: How a separation is scored

Every estimate is matched to a source (order and sign are free), then scored by the correlation rho
and by SNR = 10 log10(rho^2 / (1 - rho^2)).

In [ ]:
slide("Correlation to SNR")
rho = np.array([0.5, 0.7, 0.9, 0.99, 0.999])
snr_tab = 10 * np.log10(rho**2 / (1 - rho**2))
print(pl.DataFrame({"rho": rho, "SNR (dB)": snr_tab.round(1)}))
Xq, Sq, _ = bss.mixtures(seed=SEED)
est = FastICA(2, random_state=SEED, whiten="unit-variance", max_iter=500).fit_transform(Xq.T).T
_, rq = bss.match(Sq, est)
print(
    f"ICA on the two sines: rho {rq.round(3)}, SNR from the formula {(10 * np.log10(rq**2 / (1 - rq**2))).round(1)} dB, bss.snr_db {bss.snr_db(Sq, est).round(1)} dB"
)
assert np.allclose(10 * np.log10(rq**2 / (1 - rq**2)), bss.snr_db(Sq, est))

## Slide: Three voices, three mixtures, three methods (the ml101 cocktail party)

Three non-Gaussian signals (sine, chirp, square wave; stand-ins for three voices), three
microphones, mixing matrix $A$ with 1 on the diagonal and 0.5 elsewhere. One row per stage:
sources, mixtures, then the outputs of ICA, PCA and NMF (matched in order and sign).

In [ ]:
from scipy.optimize import linear_sum_assignment


def align(S_true: np.ndarray, E: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Order and sign of the estimates E (rows) that best match S_true (Hungarian on |corr|); also the correlations."""
    C = np.corrcoef(S_true, E)[: len(S_true), len(S_true) :]
    r, c = linear_sum_assignment(-np.abs(C))
    out = np.zeros_like(S_true, dtype=float)
    out[r] = E[c] * np.sign(C[r, c])[:, None]
    return out, np.abs(C[r, c])


n3 = 400
t3 = np.arange(n3) / n3
S3 = np.vstack(
    [np.sin(2 * np.pi * 3 * t3), np.sin(2 * np.pi * (7 * t3 + 0.8 * t3**2)), np.sign(np.sin(2 * np.pi * 5 * t3))]
)
S3 = (S3 - S3.min(1, keepdims=True)) / np.ptp(S3, axis=1, keepdims=True)  # in [0, 1]: valid for NMF without a shift
A3 = np.array([[1, 0.5, 0.5], [0.5, 1, 0.5], [0.5, 0.5, 1]])
X3 = A3 @ S3
out3 = {
    "ICA": FastICA(3, random_state=SEED, whiten="unit-variance", max_iter=1000).fit_transform(X3.T).T,
    "PCA": PCA(3).fit_transform(X3.T).T,
    "NMF": pynnmf.nmf_mu(X3, 3, n=2000, seed=SEED)[2],
}
rows3 = [("sources $S$", S3, GREY), ("mixtures $X = AS$", X3, GREY)] + [
    (m, align(S3, v)[0], c) for (m, v), c in zip(out3.items(), [ORANGE, BLUE, GREEN])
]
f, ax = plt.subplots(len(rows3), 3, figsize=(plotlib.FULL, 2.3), sharex=True)
slide("Three voices")
for r, (name, M, col) in enumerate(rows3):
    for j in range(3):
        ax[r, j].plot(t3, M[j], color=col, lw=0.7)
        ax[r, j].set_yticks([])
    ax[r, 0].set_ylabel(name, fontsize=6.5, rotation=0, ha="right", va="center")
    if r >= 2:
        cs = [abs(np.corrcoef(S3[i], M[i])[0, 1]) for i in range(3)]  # M is already aligned row by row
        for j in range(3):
            ax[r, j].text(0.99, 0.78, f"corr {cs[j]:.3f}", transform=ax[r, j].transAxes, ha="right", fontsize=6)
        print(f"{name}: correlation with the sources {np.round(cs, 3)}")
for j in range(3):
    ax[-1, j].set_xlabel("time (s)", fontsize=7)
    ax[0, j].set_title(f"signal {j + 1}", fontsize=7)
plotlib.save(f, "p5_voices")

## Slide: Three images, three mixtures, three methods (the ml101 image example)

Same experiment on images: each image is one source row of $S$ (flattened pixels),
$X = AS$ gives three mixed images. Pixel values are non-negative, so NMF needs no shift here.

In [ ]:
from sklearn.datasets import load_sample_image

side = 96


def prep(img: np.ndarray) -> np.ndarray:
    g = img.mean(axis=2) if img.ndim == 3 else img
    h, w = g.shape
    g = g[: min(h, w), : min(h, w)]
    idx = np.linspace(0, g.shape[0] - 1, side).astype(int)
    g = g[np.ix_(idx, idx)]
    return (g - g.min()) / np.ptp(g)


text = plt.figure(figsize=(1, 1), dpi=side)
text.text(0.5, 0.5, "FAA", ha="center", va="center", fontsize=34, fontweight="bold")
text.canvas.draw()
txt = 1.0 - np.asarray(text.canvas.buffer_rgba())[..., :3].mean(axis=2) / 255.0
plt.close(text)
imgs = [prep(load_sample_image("china.jpg")), prep(load_sample_image("flower.jpg")), prep(txt)]
Si = np.array([im.ravel() for im in imgs])
Xi = A3 @ Si
outi = {
    "ICA": FastICA(3, random_state=SEED, whiten="unit-variance", max_iter=1000).fit_transform(Xi.T).T,
    "PCA": PCA(3).fit_transform(Xi.T).T,
    "NMF": pynnmf.nmf_hals(Xi, 3, n=600, seed=SEED, init="nndsvd")[2],
}
rowsi = [("sources", Si), ("mixtures", Xi)] + [(m, align(Si, v)[0]) for m, v in outi.items()]
f, ax = plt.subplots(3, len(rowsi), figsize=(plotlib.FULL, 2.2))
slide("Three images")
for r, (name, M) in enumerate(rowsi):
    for j in range(3):
        m = M[j]
        ax[j, r].imshow(((m - m.min()) / (np.ptp(m) + 1e-12)).reshape(side, side), cmap="gray")
        ax[j, r].set_xticks([]), ax[j, r].set_yticks([])
        if r >= 2:
            c = abs(np.corrcoef(Si[j], M[j])[0, 1])
            ax[j, r].set_xlabel(f"corr {c:.3f}", fontsize=7.5, labelpad=1)
    ax[0, r].set_title(name, fontsize=8)
    if r >= 2:
        print(
            f"{name}: correlation with the sources {[round(float(abs(np.corrcoef(Si[j], M[j])[0, 1])), 3) for j in range(3)]}"
        )
f.subplots_adjust(hspace=0.45)
plotlib.save(f, "p5_images")

## Slide: Three real voices, three microphones (the ml101 cocktail party)

Three short voice clips (datasets/bss), zero-padded to the longest, mixed with the matrix of the
ml101 notebook (1 on the diagonal, 0.5 elsewhere), then separated by ICA, PCA and NMF.
The audio players let you listen to every row. NMF needs non-negative input: the mixtures are shifted.

In [ ]:
from IPython.display import Audio, display

from faa import data

Sv, rate = data.load_voices()
Xv = A3 @ Sv
outv = {
    "ICA": FastICA(3, random_state=SEED, whiten="unit-variance", max_iter=1000).fit_transform(Xv.T).T,
    "PCA": PCA(3).fit_transform(Xv.T).T,
    "NMF": pynnmf.nmf_mu(Xv - Xv.min() + 0.01, 3, n=1500, seed=SEED)[2],
}
rowsv = [("sources", Sv), ("mixtures", Xv)] + [(m, align(Sv, v)[0]) for m, v in outv.items()]
tv = np.arange(Sv.shape[1]) / rate
f, ax = plt.subplots(len(rowsv), 3, figsize=(plotlib.FULL, 2.0), sharex=True)
slide("Three real voices")
print(f"{Sv.shape[1]} samples at {rate} Hz = {Sv.shape[1] / rate:.2f} s")
for r, (name, M) in enumerate(rowsv):
    for j in range(3):
        ax[r, j].plot(tv, M[j] / np.abs(M[j]).max(), color=(GREY, GREY, ORANGE, BLUE, GREEN)[r], lw=0.5)
        ax[r, j].set_yticks([])
        if r >= 2:
            ax[r, j].text(
                0.99,
                0.72,
                f"corr {abs(np.corrcoef(Sv[j], M[j])[0, 1]):.3f}",
                transform=ax[r, j].transAxes,
                ha="right",
                fontsize=7.5,
            )
    ax[r, 0].set_ylabel(name, fontsize=7.5, rotation=0, ha="right", va="center")
    if r >= 2:
        print(
            f"{name}: correlation with the voices {[round(float(abs(np.corrcoef(Sv[j], M[j])[0, 1])), 3) for j in range(3)]}"
        )
for j in range(3):
    ax[-1, j].set_xlabel("time (s)", fontsize=8)
    ax[0, j].set_title(f"voice {j + 1}", fontsize=8)
plotlib.save(f, "p5_clips")
for name, M in rowsv:  # listen: the players are part of the demo
    print(name)
    for j in range(3):
        display(Audio(M[j] / np.abs(M[j]).max(), rate=rate))

## Slide: Three real pictures, three mixtures (the ml101 image example)

Three 200 x 200 pictures, one per row of S, mixed with the same matrix. Pixels are non-negative,
so NMF works without a shift.

In [ ]:
Pc = data.load_pictures()
Si = Pc.reshape(3, -1)
Xi = A3 @ Si
outi = {
    "ICA": FastICA(3, random_state=SEED, whiten="unit-variance", max_iter=1000).fit_transform(Xi.T).T,
    "PCA": PCA(3).fit_transform(Xi.T).T,
    "NMF": pynnmf.nmf_hals(Xi, 3, n=600, seed=SEED, init="nndsvd")[2],
}
rowsi = [("sources", Si), ("mixtures", Xi)] + [(m, align(Si, v)[0]) for m, v in outi.items()]
f, ax = plt.subplots(3, len(rowsi), figsize=(plotlib.FULL, 2.2))
slide("Three real pictures")
for r, (name, M) in enumerate(rowsi):
    for j in range(3):
        m = M[j]
        ax[j, r].imshow(((m - m.min()) / (np.ptp(m) + 1e-12)).reshape(200, 200), cmap="gray")
        ax[j, r].set_xticks([]), ax[j, r].set_yticks([])
        if r >= 2:
            ax[j, r].set_xlabel(f"corr {abs(np.corrcoef(Si[j], M[j])[0, 1]):.3f}", fontsize=7.5, labelpad=1)
    ax[0, r].set_title(name, fontsize=8)
    if r >= 2:
        print(
            f"{name}: correlation with the pictures {[round(float(abs(np.corrcoef(Si[j], M[j])[0, 1])), 3) for j in range(3)]}"
        )
f.subplots_adjust(hspace=0.45)
plotlib.save(f, "p5_pictures")

## Slide: More sensors help — 1, 2 and 3 mixtures of the same two sources

With one sensor the problem is ill-posed for these linear models (the second source
cannot be recovered). With three, the extra dimension absorbs noise.

In [ ]:
rows = []
for m in [1, 2, 3]:
    A_m = A[:1] if m == 1 else np.vstack([A, [0.5, 0.9]])  # 1 sensor: the first row; 3: an extra microphone
    if m == 1:
        rows.append({"sensors": m, "SNR s1 (dB)": float("nan"), "SNR s2 (dB)": float("nan")})
        continue
    Xm = A_m[:m] @ S + 0.05 * np.random.default_rng(SEED).normal(0, 1, (m, 300))
    ica_m = FastICA(m, random_state=SEED).fit(Xm.T)
    est = ica_m.transform(Xm.T).T  # (m, 300) recovered signals
    cs = [max(abs(np.corrcoef(est[j], S[i])[0, 1]) for j in range(m)) for i in range(2)]
    rows.append(
        {
            "sensors": m,
            "SNR s1 (dB)": 10 * np.log10(cs[0] ** 2 / (1 - cs[0] ** 2)),
            "SNR s2 (dB)": 10 * np.log10(cs[1] ** 2 / (1 - cs[1] ** 2)),
        }
    )
slide("Sensors sweep (ICA, two sources)")
print(pl.DataFrame(rows).with_columns(pl.col(pl.Float64).round(1)))

## Sweep and failure cases (more than the slides)

1. Gaussian sources: ICA has nothing to maximise — any orthogonal rotation is equally good.
2. NMF with the shift: the recovered parts include the offset; correlation is high but
   the amplitude is not the source's.

In [ ]:
rng = np.random.default_rng(SEED)
G = rng.normal(0, 1, (2, 300))
A_g = np.array([[0.9, 0.4], [0.7, 1.1]])
Xg = A_g @ G + 0.05 * rng.normal(0, 1, (2, 300))
ica_g = FastICA(2, random_state=SEED).fit(Xg.T)
est_g = ica_g.transform(Xg.T).T
slide("Failure: Gaussian sources")
print(f"kurtosis of the true sources: {bss.kurtosis(G).round(2)} (both ~ 0)")
print(f"ICA 'recovery' correlation:  {np.corrcoef(est_g[0], G[0])[0, 1]:.3f}, {np.corrcoef(est_g[1], G[1])[0, 1]:.3f}")
print("high correlation is luck: with Gaussian sources every rotation is a solution")

In [ ]:
slide("Failure: NMF amplitude after the shift")
print(f"true source std:  {S.std(axis=1).round(2)}")
print(f"NMF part std:     {(S_nmf - S_nmf.mean(1, keepdims=True)).std(axis=1).round(2)} (scale is arbitrary anyway)")